### Toy example: Newsvendor problem

#### Imports

In [1]:
import numpy as np
import math
from numpy.random import choice
from sklearn.utils import shuffle
import itertools
from scipy.stats import dirichlet
from tqdm import tqdm
from joblib import Parallel, delayed
import time

#### Class for sampling from the NPL posterior (Assumes a Exponential distribution model and the negative log-likelihood as the loss function)


In [5]:
# NPL class
class npl():
    """This class contains functions to perform NPL inference (for alpha = 0 in the DP prior) for the Exponential distribution model.
    The user supplies parameters:
        X: Data set
        B: number of bootstrap iterations
        p: number of unknown parameters
    """

    def __init__(self, X, B, p):
        self.B = B
        self.X = X
        self.p = p
        self.n, self.d = self.X.shape

    def draw_samples(self):
        """Draws B samples in parallel from the nonparametric posterior"""

        weights = dirichlet.rvs(np.ones(self.n), size = self.B, random_state = 13)
        wll_samples = np.zeros((self.B,self.p))

        temp = Parallel(n_jobs=-1, backend='multiprocessing', max_nbytes=None,batch_size="auto")(delayed(self.WLL)(self.X,weights[i,:]) for i in tqdm(range(self.B)))

        for i in range(self.B):
              wll_samples[i,:] = temp[i]
              self.wll_sample = np.array(wll_samples)

    def WLL(self, data, weights):
        """Get weighted negative log likelihood minimizer, for Exponential distribution model"""

        theta = np.zeros(self.d)
        for i in range(self.n):
            theta += weights[i]*data[i,:]
        return theta**(-1)

#### Sample observations from the standard Student-t ($\nu = 1$)

In [6]:
# Sample observations
degrees_of_freedom = 2
num_samples = 1000
data = np.random.standard_t(degrees_of_freedom, size=num_samples)

#### Generate NPL posterior samples

In [9]:
n = num_samples
B = 500 # number of bootstrap iterations
p = 1 # numbers of unknown parameters

npl_toy = npl(data.reshape((n,1)),B, p)
t0 = time.time()
npl_toy.draw_samples()
t1 = time.time()
total = t1-t0
print(f'Total time: {total} seconds')
wll_sample = npl_toy.wll_sample

100%|██████████| 500/500 [00:01<00:00, 339.20it/s]


Total time: 1.975379228591919 seconds


#### Calculate HPD region 